# Description

Projects S-PrediXcan tissue-specific gene-trait z-scores into the ARCHS4 CLAMP latent space using `CLAMP::projectCLAMP()`. Also saves raw S-PrediXcan results after removing NaN rows.

Mirrors `phenoplier/nbs/30_drug_disease_associations/001-spredixcan-projections.ipynb` exactly, replacing `MultiplierProjection().transform()` with `CLAMP::projectCLAMP()`.

**Input**: 49 per-tissue S-PrediXcan MASHR z-score pkl files at `data/phenomexcan/gene_assoc/spredixcan/pkl/`.  
Data source: Zenodo record 3911190 (PhenomeXcan), `spredixcan-mashr-zscores.tar`, converted from TSV.gz to pkl.

**Outputs** (`output/drug_disease_analyses/spredixcan/`):
- `raw/{stem}-data.pkl`: genes × traits, NaN rows dropped
- `proj/{stem}-projection.pkl`: LVs × traits (CLAMP projection)

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here

# Settings

In [3]:
SPREDIXCAN_FOLDER = here('data/phenomexcan/gene_assoc/spredixcan/pkl')
display(SPREDIXCAN_FOLDER)
assert SPREDIXCAN_FOLDER.exists()

CLAMP_MODEL_FILE = here('output/archs4/archs4_CLAMP_C2CP.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/phenomexcan/gene_assoc/spredixcan/pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/archs4_CLAMP_C2CP.rds')

In [4]:
DRUG_DISEASE_DIR = here('output/drug_disease_analyses')
DRUG_DISEASE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_RAW_DIR = DRUG_DISEASE_DIR / 'spredixcan' / 'raw'
OUTPUT_RAW_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_RAW_DIR)

OUTPUT_PROJ_DIR = DRUG_DISEASE_DIR / 'spredixcan' / 'proj'
OUTPUT_PROJ_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PROJ_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/raw')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/proj')

# Load CLAMP model and prepare gene mapping

In [5]:
CLAMP = importr('CLAMP')
readRDS = ro.r['readRDS']
clamp = readRDS(str(CLAMP_MODEL_FILE))
print('CLAMP model loaded')

CLAMP model loaded


In [6]:
gene_symbols = list(ro.r['rownames'](clamp.rx2('Z')))
lv_names = list(ro.r['colnames'](clamp.rx2('Z')))
print(f'CLAMP genes: {len(gene_symbols)}, LVs: {len(lv_names)}')

CLAMP genes: 18423, LVs: 2366


In [7]:
# Map CLAMP gene symbols (HGNC) → Ensembl IDs
# S-PrediXcan files use Ensembl IDs as index.
clusterProfiler = importr('clusterProfiler')

bitr_result = clusterProfiler.bitr(
    ro.StrVector(gene_symbols),
    fromType='SYMBOL',
    toType='ENSEMBL',
    OrgDb='org.Hs.eg.db',
)

with localconverter(ro.default_converter + pandas2ri.converter):
    mapping_df = ro.conversion.rpy2py(bitr_result)

print(f'Raw mapping shape: {mapping_df.shape}')
display(mapping_df.head())

R callback write-console: 
  


R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (19313, 2)


,SYMBOL,ENSEMBL
1,A1BG,ENSG00000121410
2,A1BG-AS1,ENSG00000268895
3,A2M,ENSG00000175899
4,A2M-AS1,ENSG00000245105
5,A2ML1,ENSG00000166535


In [8]:
# Keep only 1:1 unambiguous symbol ↔ Ensembl mappings
dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')
print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')

mapped_symbols = mapping_1to1.index.tolist()
mapped_ensembl = mapping_1to1['ENSEMBL'].tolist()

1:1 mappings: 16038 / 18423 CLAMP genes


In [9]:
# Build CLAMP sub-object with Z restricted to 1:1-mapped genes.
# as.matrix() ensures dense numeric matrix for projectCLAMP's %*% operator.
subset_Z = ro.r('function(clamp, genes) { clamp$Z <- as.matrix(clamp$Z[genes, ]); clamp }')
clamp_sub = subset_Z(clamp, ro.StrVector(mapped_symbols))
print(f'Subsetted CLAMP Z: {len(mapped_symbols)} genes x {len(lv_names)} LVs')

R callback write-console: In addition:   


R callback write-console: Warning message:
  


R callback write-console: In (function (geneID, fromType, toType, OrgDb, drop = TRUE)  :  


R callback write-console: 
   


R callback write-console:  6.63% of input gene IDs are fail to map...
  


Subsetted CLAMP Z: 16038 genes x 2366 LVs


# Load and project S-PrediXcan tissue files

In [10]:
input_file_list = sorted(SPREDIXCAN_FOLDER.glob('spredixcan-mashr-zscores-*.pkl'))
_tmp = len(input_file_list)
display(_tmp)
assert _tmp == 49, f'Expected 49 tissue files, found {_tmp}'

49

In [11]:
for input_file in input_file_list:
    print(input_file.name)

    # --- load ---
    data = pd.read_pickle(input_file)
    print(f'  shape: {data.shape}')

    # Drop duplicate gene indices (rare: ~4 genes across all tissues)
    n_dups = data.index.duplicated().sum()
    if n_dups > 0:
        print(f'  dropping {n_dups} duplicate gene rows')
        data = data[~data.index.duplicated(keep='first')]

    assert data.index.is_unique
    assert data.columns.is_unique

    # Drop genes with any NaN across traits (same as PhenoPlier)
    data = data.dropna(how='any')
    print(f'  shape (no NaN, no dups): {data.shape}')
    assert not data.isna().any().any()

    # --- save raw ---
    output_raw = OUTPUT_RAW_DIR / f'{input_file.stem}-data.pkl'
    print(f'  saving raw to: {output_raw}')
    data.to_pickle(output_raw)

    # --- project through CLAMP ---
    print('  projecting through CLAMP...')

    # Align to CLAMP gene order (mapped Ensembl IDs), fill missing with 0
    aligned = data.reindex(mapped_ensembl).fillna(0.0).values  # (n_genes, n_traits)

    r_mat = ro.r['matrix'](
        ro.FloatVector(aligned.flatten('F')),
        nrow=aligned.shape[0],
        ncol=aligned.shape[1],
    )

    proj_r = CLAMP.projectCLAMP(clamp_sub, newdata=r_mat)

    with localconverter(ro.default_converter + pandas2ri.converter):
        proj_values = ro.conversion.rpy2py(proj_r)

    projection = pd.DataFrame(proj_values, index=lv_names, columns=data.columns)
    print(f'    projection shape: {projection.shape}')

    # --- save projection ---
    output_proj = OUTPUT_PROJ_DIR / f'{input_file.stem}-projection.pkl'
    print(f'    saving projection to: {output_proj}')
    projection.to_pickle(output_proj)

    print('')

spredixcan-mashr-zscores-Adipose_Subcutaneous.pkl


  shape: (14720, 4091)
  dropping 4 duplicate gene rows


  shape (no NaN, no dups): (14059, 4091)
  saving raw to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/raw/spredixcan-mashr-zscores-Adipose_Subcutaneous-data.pkl


  projecting through CLAMP...


    projection shape: (2366, 4091)
    saving projection to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/proj/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection.pkl

spredixcan-mashr-zscores-Adipose_Visceral_Omentum.pkl


...

    projection shape: (2366, 4091)
    saving projection to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/proj/spredixcan-mashr-zscores-Vagina-projection.pkl

spredixcan-mashr-zscores-Whole_Blood.pkl


  shape: (12610, 4091)
  dropping 1 duplicate gene rows


  shape (no NaN, no dups): (12066, 4091)
  saving raw to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/raw/spredixcan-mashr-zscores-Whole_Blood-data.pkl


  projecting through CLAMP...


    projection shape: (2366, 4091)
    saving projection to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/proj/spredixcan-mashr-zscores-Whole_Blood-projection.pkl

